# Deployment
## Studi Kasus: Iris Flower Dataset

---

## 6.1 Konsep Deployment dalam Praktik Industri

Setelah model lulus tahap Evaluasi dengan nilai akurasi yang memuaskan, langkah pemungkas dalam CRISP-DM adalah **Deployment** (Penerapan). 

Di dunia industri nyata, Deployment berarti kita mengemas model analitik menjadi sebuah layanan (*service*), seperti API, *Web Application*, atau aplikasi *mobile*. Tujuannya agar end-user (misal: peneliti biologi) bisa memanfaatkan kecerdasan model ini tanpa harus paham kode pemrograman. 

Pada modul terakhir ini, saya merancang sebuah simulasi sistem *Deployment* sederhana untuk mendemonstrasikan bagaimana data mentah dari *user* akan mengalir, ditransformasi, diprediksi, dan diterjemahkan kembali menjadi bahasa manusia.

## 6.2 Import Library dan Inisiasi Sistem
Pertama-tama, saya muat kembali model `knn_model.pkl` yang bertindak sebagai "otak" utama sistem kita.

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

# Memuat 'otak' AI kita
model = joblib.load('knn_model.pkl')
print('[Sistem Online] Model KNN berhasil dimuat ke dalam memori.')

[Sistem Online] Model KNN berhasil dimuat ke dalam memori.


## 6.3 Rekonstruksi Skala (Kunci Keberhasilan Prediksi!)

Ada satu konsep **krusial** di sini. Ingat bahwa di tahap *Data Preparation*, model kita dilatih menggunakan data yang sudah distandarisasi (*Standard Scaler*). 

Ini berarti, model kita beroperasi pada angka-angka berskala (misal -1 sampai +1). Jika seorang ahli botani tiba-tiba menginput angka `5.1` (ukuran kelopak nyata), model akan kebingungan dan memberikan prediksi ngawur. Oleh karena itu, kita **WAJIB** men-*scale* data input baru tersebut menggunakan parameter rasio yang persis sama dengan data latih aslinya.

Dalam skenario ini, saya merekonstruksi objek `scaler` berdasarkan data asli untuk memproses input pengguna nanti.

In [2]:
df_raw = pd.read_csv('IRIS.csv')
scaler = StandardScaler()
# Pelajari rentang skala dari data historis
scaler.fit(df_raw[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']])
print('[Sistem Online] Mesin standarisasi (Scaler) berhasil disiapkan.')

[Sistem Online] Mesin standarisasi (Scaler) berhasil disiapkan.


## 6.4 Membangun Fungsi Prediksi (Simulasi Backend API)

Berikutnya, saya menyusun fungsi `prediksi_spesies_iris`. Fungsi ini bekerja otomatis seperti pelayan (*server backend*) yang melakukan 4 tahap beruntun:
1. **Data Ingestion**: Menangkap input 4 ukuran bunga dari pengguna.
2. **Preprocessing**: Mentransformasikan ukuran asli menjadi format *scaled*.
3. **Inference**: Menyuruh model KNN melakukan prediksi angka (0, 1, atau 2).
4. **Postprocessing**: Mengubah angka (0,1,2) kembali menjadi kata-kata (Setosa/Versicolor/Virginica) agar mudah dibaca manusia.

In [3]:
def prediksi_spesies_iris(sepal_l, sepal_w, petal_l, petal_w):
    # 1. Konversi ke array 2 dimensi agar bisa dibaca model
    input_data = np.array([[sepal_l, sepal_w, petal_l, petal_w]])
    
    # 2. Skala input data agar senada dengan pelatihan model
    input_scaled = scaler.transform(input_data)
    
    # 3. Eksekusi prediksi!
    prediksi_encoded = model.predict(input_scaled)[0]
    
    # 4. Decode label untuk output akhir pengguna
    label_map = {0: 'Iris-setosa', 1: 'Iris-versicolor', 2: 'Iris-virginica'}
    hasil_spesies = label_map[prediksi_encoded]
    
    return hasil_spesies

## 6.5 Simulasi Kasus Nyata (Uji Coba Lapangan)

Mari kita mendemonstrasikannya! Bayangkan saya seorang Botanis yang sedang berada di lapangan. Saya mengukur bunga baru dengan spesifikasi:
- Panjang Sepal: 6.5 cm
- Lebar Sepal: 3.0 cm
- Panjang Petal: 5.2 cm
- Lebar Petal: 2.0 cm

Saya akan menyerahkan angka ini kepada fungsi (API) sistem kita.

In [4]:
bunga_1 = prediksi_spesies_iris(sepal_l=6.5, sepal_w=3.0, petal_l=5.2, petal_w=2.0)
print(f'Bunga pertama diidentifikasi sebagai spesies: >>> {bunga_1} <<<')

# Skenario Bunga Kedua
bunga_2 = prediksi_spesies_iris(sepal_l=5.1, sepal_w=3.5, petal_l=1.4, petal_w=0.2)
print(f'Bunga kedua diidentifikasi sebagai spesies:   >>> {bunga_2} <<<')

Bunga pertama diidentifikasi sebagai spesies: >>> Iris-virginica <<<
Bunga kedua diidentifikasi sebagai spesies:   >>> Iris-setosa <<<


c:\Users\Surya Maulana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Surya Maulana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\Surya Maulana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Surya Maulana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


## 6.6 Penutup: Akhir Siklus CRISP-DM

Melalui eksekusi fungsi deployment ini, sistem yang telah kita bangun secara resmi terbukti siap operasional. Proses panjang Penambangan Data mulai dari memahami tujuan (*Business Understanding*) hingga menerapkan kecerdasan di dunia nyata (*Deployment*) telah tuntas dengan hasil yang solid dan terstruktur. 

Dengan sistem arsitektur seperti ini, jika kelak kita memiliki ratusan data input baru setiap detiknya, model dapat memprosesnya dengan instan dan tingkat keakuratan yang dapat diandalkan.